<a href="https://colab.research.google.com/github/hamnasz/DevByte/blob/main/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python Pandas Data Cleanup

**DevByte AI/ML Internship — BETA Project**

This notebook loads a raw CSV dataset, cleans null entries, normalizes column names and numeric values, and parses date-range columns into structured start/end dates.

**Pipeline steps:**
1. Load raw CSV (a sample dataset is generated below so this notebook runs end-to-end in Colab without any upload — swap it for your real file in Step 1)
2. Inspect data quality
3. Normalize column names
4. Clean null entries
5. Normalize numeric column values
6. Parse date-range strings into start/end dates
7. Export the cleaned dataset

In [ ]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)

## 1. Load the Raw CSV

No specific dataset was provided for this task, so the cell below generates a synthetic CSV with realistic data-quality issues like missing values, messy column headers, and date-range strings purely so the rest of the notebook is runnable as-is.

#### Sample dataset generation (remove this cell when using a real CSV)

In [ ]:
import random
random.seed(42)
np.random.seed(42)

In [ ]:
n = 200
regions = ['North', 'South', 'East', 'West', None]
categories = ['Electronics', 'Apparel', 'Grocery', 'Furniture']

In [ ]:
def random_date_range():
    start = pd.Timestamp('2023-01-01') + pd.Timedelta(days=random.randint(0, 300))
    end = start + pd.Timedelta(days=random.randint(1, 30))
    return f"{start.date()} - {end.date()}"

In [ ]:
data = {
    ' Customer ID ': range(1001, 1001 + n),
    'Customer Name': [f"Customer_{i}" if random.random() > 0.05 else None for i in range(n)],
    'Region ': [random.choice(regions) for _ in range(n)],
    'Product Category': [random.choice(categories) for _ in range(n)],
    'Order Amount (PKR)': [round(random.uniform(500, 50000), 2) if random.random() > 0.08 else np.nan for _ in range(n)],
    'Quantity': [random.randint(1, 20) if random.random() > 0.05 else np.nan for _ in range(n)],
    'Order Date Range': [random_date_range() for _ in range(n)],
}

In [ ]:
raw_df = pd.DataFrame(data)
raw_df.to_csv('raw_data.csv', index=False)
print('Sample raw_data.csv created with shape:', raw_df.shape)
raw_df.head()

Sample raw_data.csv created with shape: (200, 7)


,Customer ID,Customer Name,Region,Product Category,Order Amount (PKR),Quantity,Order Date Range
0,1001,Customer_0,North,Apparel,26326.46,9.0,2023-04-09 - 2023-04-17
1,1002,None,South,Grocery,30987.66,12.0,2023-03-12 - 2023-03-17
2,1003,Customer_2,South,Electronics,38844.75,8.0,2023-02-09 - 2023-02-19
3,1004,Customer_3,West,Furniture,41373.86,20.0,2023-02-21 - 2023-03-10
4,1005,Customer_4,West,Electronics,16454.08,18.0,2023-10-04 - 2023-10-31


In [ ]:
df = pd.read_csv('raw_data.csv')

In [ ]:
print('Shape:', df.shape)

Shape: (200, 7)


In [ ]:
df.head()

,Customer ID,Customer Name,Region,Product Category,Order Amount (PKR),Quantity,Order Date Range
0,1001,Customer_0,North,Apparel,26326.46,9.0,2023-04-09 - 2023-04-17
1,1002,NaN,South,Grocery,30987.66,12.0,2023-03-12 - 2023-03-17
2,1003,Customer_2,South,Electronics,38844.75,8.0,2023-02-09 - 2023-02-19
3,1004,Customer_3,West,Furniture,41373.86,20.0,2023-02-21 - 2023-03-10
4,1005,Customer_4,West,Electronics,16454.08,18.0,2023-10-04 - 2023-10-31


## 2. Inspect Data Quality

Always profile a raw file before touching it

In [ ]:
print('Raw column names:')

Raw column names:


In [ ]:
print(list(df.columns))

[' Customer ID ', 'Customer Name', 'Region ', 'Product Category', 'Order Amount (PKR)', 'Quantity', 'Order Date Range']


In [ ]:
print('\nData types:\n', df.dtypes)


Data types:
  Customer ID            int64
Customer Name          object
Region                 object
Product Category       object
Order Amount (PKR)    float64
Quantity              float64
Order Date Range       object
dtype: object


In [ ]:
print('\nMissing values per column:\n', df.isnull().sum())


Missing values per column:
  Customer ID           0
Customer Name         10
Region                39
Product Category       0
Order Amount (PKR)    12
Quantity              10
Order Date Range       0
dtype: int64


In [ ]:
print(f"\nTotal missing cells: {df.isnull().sum().sum()} out of {df.size}")


Total missing cells: 71 out of 1400


## 3. Normalize Column Names

Raw CSVs often have inconsistent headers like extra spaces, mixed case, special characters. Standardize everything to clean `snake_case` so downstream code can reference columns reliably.

In [ ]:
def normalize_column_name(col):
    col = col.strip().lower()
    col = re.sub(r'[^\w\s]', '', col)   # strip special characters like ( )
    col = re.sub(r'\s+', '_', col)      # collapse whitespace into underscores
    return col

In [ ]:
df.columns = [normalize_column_name(c) for c in df.columns]

In [ ]:
print('Normalized columns:', list(df.columns))

Normalized columns: ['customer_id', 'customer_name', 'region', 'product_category', 'order_amount_pkr', 'quantity', 'order_date_range']


In [ ]:
df.head()

,customer_id,customer_name,region,product_category,order_amount_pkr,quantity,order_date_range
0,1001,Customer_0,North,Apparel,26326.46,9.0,2023-04-09 - 2023-04-17
1,1002,NaN,South,Grocery,30987.66,12.0,2023-03-12 - 2023-03-17
2,1003,Customer_2,South,Electronics,38844.75,8.0,2023-02-09 - 2023-02-19
3,1004,Customer_3,West,Furniture,41373.86,20.0,2023-02-21 - 2023-03-10
4,1005,Customer_4,West,Electronics,16454.08,18.0,2023-10-04 - 2023-10-31


## 4. Clean Null Entries

Strategy used here (adjust to fit your actual dataset):
- Drop rows missing the primary identifier (`customer_id`) — these can't be meaningfully imputed.
- Fill missing categorical/text values with `'Unknown'`.
- Fill missing numeric values with the column median (robust to outliers, unlike the mean).

In [ ]:
df_clean = df.copy()

In [ ]:
print('Missing values before cleaning:\n', df_clean.isnull().sum())

Missing values before cleaning:
 customer_id          0
customer_name       10
region              39
product_category     0
order_amount_pkr    12
quantity            10
order_date_range     0
dtype: int64


#### Drop rows missing the primary key

In [ ]:
df_clean = df_clean.dropna(subset=['customer_id'])

#### Fill missing categorical/text values

In [ ]:
categorical_cols = df_clean.select_dtypes(include='object').columns
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna('Unknown')

#### Fill missing numeric values with the column median

In [ ]:
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

In [ ]:
print('\nMissing values after cleaning:\n', df_clean.isnull().sum())


Missing values after cleaning:
 customer_id         0
customer_name       0
region              0
product_category    0
order_amount_pkr    0
quantity            0
order_date_range    0
dtype: int64


In [ ]:
print('Shape after cleaning:', df_clean.shape)

Shape after cleaning: (200, 7)


## 5. Normalize Numeric Column Values

Min-max scale numeric columns into a common 0–1 range so they're directly comparable across different magnitudes (e.g. `order_amount_pkr` in thousands vs `quantity` in single digits).

In [ ]:
def min_max_normalize(series):
    return (series - series.min()) / (series.max() - series.min())

In [ ]:
cols_to_normalize = ['order_amount_pkr', 'quantity']

In [ ]:
for col in cols_to_normalize:
    if col in df_clean.columns:
        df_clean[f'{col}_normalized'] = min_max_normalize(df_clean[col])

In [ ]:
preview_cols = cols_to_normalize + [f'{c}_normalized' for c in cols_to_normalize]

In [ ]:
df_clean[preview_cols].describe()

,order_amount_pkr,quantity,order_amount_pkr_normalized,quantity_normalized
count,200.000000,200.000000,200.000000,200.000000
mean,25232.712200,10.880000,0.498677,0.520000
std,13663.608162,5.801196,0.276916,0.305326
min,626.980000,1.000000,0.000000,0.000000
25%,14456.695000,5.750000,0.280283,0.250000
50%,24518.565000,11.000000,0.484204,0.526316
75%,35711.082500,16.000000,0.711039,0.789474
max,49968.990000,20.000000,1.000000,1.000000


## 6. Parse Date Ranges

`order_date_range` stores a range as one string (e.g. `2023-03-01 - 2023-03-15`). Split it into proper `start_date` / `end_date` datetime columns and derive `duration_days`.

In [ ]:
split_dates = df_clean['order_date_range'].str.split(' - ', expand=True)

In [ ]:
df_clean['start_date'] = pd.to_datetime(split_dates[0], errors='coerce')

In [ ]:
df_clean['end_date'] = pd.to_datetime(split_dates[1], errors='coerce')

In [ ]:
df_clean['duration_days'] = (df_clean['end_date'] - df_clean['start_date']).dt.days

In [ ]:
unparsed = df_clean['start_date'].isnull().sum()

In [ ]:
print(f'Rows where the date range failed to parse: {unparsed}')

Rows where the date range failed to parse: 0


In [ ]:
df_clean[['order_date_range', 'start_date', 'end_date', 'duration_days']].head()

,order_date_range,start_date,end_date,duration_days
0,2023-04-09 - 2023-04-17,2023-04-09,2023-04-17,8
1,2023-03-12 - 2023-03-17,2023-03-12,2023-03-17,5
2,2023-02-09 - 2023-02-19,2023-02-09,2023-02-19,10
3,2023-02-21 - 2023-03-10,2023-02-21,2023-03-10,17
4,2023-10-04 - 2023-10-31,2023-10-04,2023-10-31,27


## 7. Final Review & Export

In [ ]:
print('Final cleaned dataset shape:', df_clean.shape)

Final cleaned dataset shape: (200, 12)


In [ ]:
print('\nFinal columns and dtypes:\n')


Final columns and dtypes:



In [ ]:
print(df_clean.dtypes)

customer_id                             int64
customer_name                          object
region                                 object
product_category                       object
order_amount_pkr                      float64
quantity                              float64
order_date_range                       object
order_amount_pkr_normalized           float64
quantity_normalized                   float64
start_date                     datetime64[ns]
end_date                       datetime64[ns]
duration_days                           int64
dtype: object


In [ ]:
df_clean.to_csv('cleaned_data.csv', index=False)

In [ ]:
print("\nSaved cleaned_data.csv")


Saved cleaned_data.csv


In [ ]:
df_clean.head(10)

,customer_id,customer_name,region,product_category,order_amount_pkr,quantity,order_date_range,order_amount_pkr_normalized,quantity_normalized,start_date,end_date,duration_days
0,1001,Customer_0,North,Apparel,26326.46,9.0,2023-04-09 - 2023-04-17,0.520844,0.421053,2023-04-09,2023-04-17,8
1,1002,Unknown,South,Grocery,30987.66,12.0,2023-03-12 - 2023-03-17,0.615311,0.578947,2023-03-12,2023-03-17,5
2,1003,Customer_2,South,Electronics,38844.75,8.0,2023-02-09 - 2023-02-19,0.774548,0.368421,2023-02-09,2023-02-19,10
3,1004,Customer_3,West,Furniture,41373.86,20.0,2023-02-21 - 2023-03-10,0.825805,1.000000,2023-02-21,2023-03-10,17
4,1005,Customer_4,West,Electronics,16454.08,18.0,2023-10-04 - 2023-10-31,0.320763,0.894737,2023-10-04,2023-10-31,27
5,1006,Customer_5,West,Apparel,46763.78,20.0,2023-09-27 - 2023-09-29,0.935041,1.000000,2023-09-27,2023-09-29,2
6,1007,Customer_6,South,Grocery,22814.70,3.0,2023-06-22 - 2023-07-21,0.449672,0.105263,2023-06-22,2023-07-21,29
7,1008,Customer_7,West,Furniture,28707.56,15.0,2023-03-09 - 2023-03-29,0.569101,0.736842,2023-03-09,2023-03-29,20
8,1009,Customer_8,North,Electronics,17151.63,10.0,2023-07-12 - 2023-07-17,0.334900,0.473684,2023-07-12,2023-07-17,5
9,1010,Unknown,South,Grocery,16588.20,4.0,2023-03-25 - 2023-03-31,0.323481,0.157895,2023-03-25,2023-03-31,6


### Downloading the result in Colab

In [ ]:
from google.colab import files
files.download('cleaned_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>